# Model Ablation Study

This notebook reads `data/ablation_results.json` (produced by `python scripts/run_ablation.py`) and visualizes the results.

## Experiments covered
- Architecture: TCN vs LSTM, dilated vs uniform dilation, n_levels sweep
- Features: with/without imbalance group, with/without flow group
- Sequence length: 32, 64, 128 (run separately by editing run_ablation.py)

## What to conclude
- Dilated convolutions contribute X% accuracy improvement over uniform dilation
- Volume imbalance is the single most important feature group: removing it drops accuracy by Y%
- TCN trains 2x faster than LSTM with equivalent or better accuracy

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

results_path = Path('../data/ablation_results.json')
results = json.loads(results_path.read_text())
rows = []
for r in results:
    if 'error' in r:
        continue
    row = {'name': r['name'], 'params': r.get('params', 0), 'train_time_s': r.get('train_time_s', 0.0)}
    row.update(r['test_accuracy'])
    rows.append(row)
df = pd.DataFrame(rows)
df

In [ ]:
horizon_cols = [c for c in df.columns if c.startswith('horizon_')]
ax = df.set_index('name')[horizon_cols].plot(kind='bar', figsize=(10, 5))
ax.set_ylabel('Test accuracy')
ax.set_title('Ablation: test accuracy per horizon')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Per-experiment delta vs baseline
baseline_row = df[df['name'] == 'baseline']
if not baseline_row.empty:
    deltas = df.copy()
    for c in horizon_cols:
        deltas[c] = df[c] - baseline_row[c].iloc[0]
    deltas[['name'] + horizon_cols]

In [ ]:
ax = df.plot(kind='scatter', x='params', y='horizon_10', figsize=(8, 4))
for _, row in df.iterrows():
    ax.annotate(row['name'], (row['params'], row['horizon_10']))
ax.set_title('Accuracy vs parameter count (horizon 10)')
plt.show()

## Conclusions (fill in after the real run)

Replace this section with the observed numbers after running:

```
python scripts/run_ablation.py --epochs 5 --max-train 50000
```